In [ ]:
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    # Test thật để chắc chắn không bị lỗi kernel
    try:
        x = torch.zeros(1, device='cuda') + 1
        print('GPU OK')
    except Exception as e:
        print(f'GPU lỗi: {e}')
        device = torch.device('cpu')
        print('Fallback CPU')

In [ ]:
import struct, io, os, random, shutil
from pathlib import Path
from PIL import Image, ImageFile
from tqdm import tqdm
ImageFile.LOAD_TRUNCATED_IMAGES = True

SEED = 42
TRAIN_SPLIT = 0.7
random.seed(SEED)

REC_PATH  = '/kaggle/input/datasets/debarghamitraroy/casia-webface/casia-webface/train.rec'
LST_PATH  = '/kaggle/input/datasets/debarghamitraroy/casia-webface/casia-webface/train.lst'
TRAIN_DIR = Path('/kaggle/working/CASIA-WebFace/train')
TEST_DIR  = Path('/kaggle/working/CASIA-WebFace/test')
TRAIN_DIR.mkdir(parents=True, exist_ok=True)
TEST_DIR.mkdir(parents=True, exist_ok=True)

print('Parsing LST...')
lst_entries = []
identity_set = set()
with open(LST_PATH, 'rb') as f:
    for line in f:
        line = line.decode('utf-8', errors='ignore').strip()
        if not line: continue
        parts = line.split('\t')
        for p in parts:
            if '.jpg' in p or '.png' in p:
                pp = p.replace('\\','/').split('/')
                if len(pp) >= 2:
                    lst_entries.append((pp[-2], pp[-1]))
                    identity_set.add(pp[-2])
                break
print(f'LST: {len(lst_entries):,} entries | {len(identity_set):,} identities')

all_ids = sorted(identity_set)
random.shuffle(all_ids)
split = int(len(all_ids) * TRAIN_SPLIT)
train_ids = set(all_ids[:split])
test_ids  = set(all_ids[split:])
print(f'Train identities: {len(train_ids):,} | Test identities: {len(test_ids):,}')

MAGIC = b'\x0a\x23\xd7\xce'

def stream_records(path):
    """Đọc tuần tự bằng file.read() trực tiếp theo offset — không slice buffer lớn."""
    with open(path, 'rb') as f:
        # Đọc magic + length liên tục, KHÔNG giữ buffer tích lũy
        while True:
            magic = f.read(4)
            if len(magic) < 4:
                break
            if magic != MAGIC:
                # Lệch — đọc từng byte tìm lại magic (hiếm khi xảy ra)
                f.seek(-3, os.SEEK_CUR)
                continue
            length_bytes = f.read(4)
            if len(length_bytes) < 4:
                break
            length = struct.unpack('<I', length_bytes)[0]
            rest = f.read(length - 4)   # đọc thẳng số byte cần, không buffer dư
            if len(rest) < length - 4:
                break
            yield rest[24:]   # bỏ 24-byte IRHeader, phần còn lại là JPEG

ok = err = skip = 0
for i, img_bytes in enumerate(tqdm(stream_records(REC_PATH), total=len(lst_entries), desc='Extracting')):
    if i >= len(lst_entries):
        break
    identity, filename = lst_entries[i]
    if identity not in identity_set:
        continue

    dest_root = TRAIN_DIR if identity in train_ids else TEST_DIR
    out_folder = dest_root / identity
    out_path   = out_folder / filename

    if out_path.exists():
        skip += 1
        continue

    try:
        img = Image.open(io.BytesIO(img_bytes)).convert('RGB')
        out_folder.mkdir(exist_ok=True)
        img.save(str(out_path), 'JPEG', quality=95)
        img.close()
        ok += 1
    except Exception:
        err += 1

    if (i + 1) % 50_000 == 0:
        print(f'  [{i+1:,}] OK={ok:,} Err={err:,} Skip={skip:,}')

print(f'\nDone! OK={ok:,} | Err={err:,} | Skip={skip:,}')
print(f'Train identities on disk: {len(list(TRAIN_DIR.iterdir())):,}')
print(f'Test  identities on disk: {len(list(TEST_DIR.iterdir())):,}')

In [ ]:
import subprocess, sys, urllib.request, os
subprocess.run([sys.executable,'-m','pip','install','facenet-pytorch==2.5.3','-q'], check=True)

import facenet_pytorch
data_dir = os.path.join(os.path.dirname(facenet_pytorch.__file__), 'data')
os.makedirs(data_dir, exist_ok=True)
BASE = 'https://github.com/timesler/facenet-pytorch/releases/download/v2.2.9/'
for w in ['pnet.pt','rnet.pt','onet.pt','20180402-114759-vggface2.pt']:
    dst = os.path.join(data_dir, w)
    if not os.path.exists(dst):
        urllib.request.urlretrieve(BASE+w, dst)
        print(f'Downloaded {w}')
    else:
        print(f'OK {w}')

In [ ]:
import os, random, time, warnings, io
warnings.filterwarnings('ignore')
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True
import numpy as np
from tqdm import tqdm
from pathlib import Path
import torch, torch.nn as nn, torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from sklearn.metrics import roc_curve, auc, accuracy_score
import matplotlib.pyplot as plt
from facenet_pytorch import InceptionResnetV1

# ── Constants ─────────────────────────────────────────────────────────────────
SEED          = 42
IMG_SIZE      = 160
BATCH_SIZE    = 64
LR            = 5e-5
WD            = 1e-4
NUM_EPOCHS    = 20
MARGIN        = 0.5
NUM_TRIPLETS  = 100_000
NUM_TEST_PAIRS = 3000
NUM_WORKERS   = 2

random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

# device đã detect ở Cell 1 — nếu chạy độc lập thì detect lại
if 'device' not in dir():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

# ── Paths — đọc thẳng từ train/ và test/ đã chia sẵn ──────────────────────────
DATA_DIR   = Path('/kaggle/working/CASIA-WebFace')
TRAIN_DIR  = DATA_DIR / 'train'
TEST_DIR   = DATA_DIR / 'test'
CHECKPOINT = Path('/kaggle/working/checkpoints'); CHECKPOINT.mkdir(exist_ok=True)
PLOTS_DIR  = Path('/kaggle/working/plots');       PLOTS_DIR.mkdir(exist_ok=True)

def get_identity_map(root):
    m = {}
    for d in sorted(root.iterdir()):
        if not d.is_dir(): continue
        imgs = [str(p) for p in sorted(d.iterdir()) if p.suffix.lower() in ('.jpg','.jpeg','.png')]
        if imgs: m[d.name] = imgs
    return m

train_map = get_identity_map(TRAIN_DIR)
test_map  = get_identity_map(TEST_DIR)
print(f'Train identities: {len(train_map):,} | Test identities: {len(test_map):,}')
print(f'Train images: {sum(len(v) for v in train_map.values()):,} | Test images: {sum(len(v) for v in test_map.values()):,}')

# ── Transforms ────────────────────────────────────────────────────────────────
train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.ColorJitter(0.2, 0.2, 0.1),
    transforms.RandomGrayscale(0.05),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])
test_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

# ── Datasets ──────────────────────────────────────────────────────────────────
class TripletDataset(Dataset):
    def __init__(self, imap, n=100000, tf=None):
        self.imap = {k: v for k, v in imap.items() if len(v) >= 2}
        self.ids  = list(self.imap.keys())
        self.n    = n
        self.tf   = tf
        self.trip = self._gen()
    def _gen(self):
        t = []
        for _ in range(self.n):
            pid = random.choice(self.ids)
            a, p = random.sample(self.imap[pid], 2)
            nid = random.choice([i for i in self.ids if i != pid])
            t.append((a, p, random.choice(self.imap[nid])))
        return t
    def _load(self, path):
        try:
            img = Image.open(path).convert('RGB')
        except Exception:
            img = Image.new('RGB', (IMG_SIZE, IMG_SIZE), 0)
        return self.tf(img) if self.tf else img
    def __len__(self): return self.n
    def __getitem__(self, i):
        a, p, n = self.trip[i]
        return self._load(a), self._load(p), self._load(n)

class PairDataset(Dataset):
    """Dùng cho cả validation (trong lúc train) và test cuối cùng."""
    def __init__(self, imap, n=3000, tf=None):
        multi = {k: v for k, v in imap.items() if len(v) >= 2}
        aids  = list(imap.keys())
        self.pairs, self.labels, self.tf = [], [], tf
        for _ in range(n // 2):
            pid = random.choice(list(multi.keys()))
            a, b = random.sample(multi[pid], 2)
            self.pairs.append((a, b)); self.labels.append(1)
        for _ in range(n // 2):
            i1, i2 = random.sample(aids, 2)
            self.pairs.append((random.choice(imap[i1]), random.choice(imap[i2])))
            self.labels.append(0)
    def __len__(self): return len(self.pairs)
    def __getitem__(self, i):
        a, b = self.pairs[i]
        try:
            a = Image.open(a).convert('RGB'); b = Image.open(b).convert('RGB')
        except Exception:
            a = b = Image.new('RGB', (IMG_SIZE, IMG_SIZE), 0)
        if self.tf: a, b = self.tf(a), self.tf(b)
        return a, b, torch.tensor(self.labels[i], dtype=torch.float32)

train_ds = TripletDataset(train_map, NUM_TRIPLETS, train_tf)
# Validation trong lúc train cũng lấy từ train_map (giữ lại 1 phần) — ở đây dùng test_map làm validation luôn
val_ds   = PairDataset(test_map, NUM_TEST_PAIRS, test_tf)

_pin = device.type == 'cuda'
train_loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=_pin,
                          drop_last=True, persistent_workers=True, prefetch_factor=2)
val_loader   = DataLoader(val_ds, BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=_pin,
                          persistent_workers=True, prefetch_factor=2)
print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')

# ── Model ─────────────────────────────────────────────────────────────────────
class FaceNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.bb = InceptionResnetV1(pretrained='vggface2', classify=False)
    def forward(self, x):
        return F.normalize(self.bb(x), p=2, dim=1)

model = FaceNet().to(device)

ckpt_path = CHECKPOINT / 'facenet_best.pth'
if ckpt_path.exists():
    model.load_state_dict(torch.load(ckpt_path, map_location=device)['model_state_dict'])
    print('Loaded previous checkpoint')

for name, p in model.named_parameters():
    p.requires_grad = any(k in name for k in ['block7', 'block8', 'last_linear', 'last_bn'])
print(f'Trainable: {sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6:.1f}M')

# ── Loss / Optimizer ──────────────────────────────────────────────────────────
class TripletLoss(nn.Module):
    def __init__(self, m=0.5): super().__init__(); self.m = m
    def forward(self, a, p, n):
        l = F.relu((a-p).pow(2).sum(1) - (a-n).pow(2).sum(1) + self.m)
        return l.mean(), (l > 0).float().mean().item()

criterion = TripletLoss(MARGIN)
optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LR, weight_decay=WD)

def lr_lambda(epoch):
    if epoch < 2: return (epoch + 1) / 2
    return 0.5 * (1 + np.cos(np.pi * (epoch - 2) / (NUM_EPOCHS - 2)))
scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

use_amp = device.type == 'cuda'
scaler  = torch.cuda.amp.GradScaler(enabled=use_amp)

def evaluate(model, loader):
    model.eval(); dists, labs = [], []
    with torch.no_grad():
        for i1, i2, lb in loader:
            i1, i2 = i1.to(device), i2.to(device)
            with torch.cuda.amp.autocast(enabled=use_amp):
                e1 = model(i1); e2 = model(i2)
            dists.extend((e1 - e2).pow(2).sum(1).sqrt().cpu().tolist())
            labs.extend(lb.tolist())
    dists = np.array(dists); labs = np.array(labs)
    fpr, tpr, thr = roc_curve(labs, -dists)
    thr_opt = -thr[np.argmax(tpr - fpr)]
    return accuracy_score(labs, (dists < thr_opt).astype(int)), auc(fpr, tpr), thr_opt

# ── Training loop ─────────────────────────────────────────────────────────────
history = {'loss': [], 'val_acc': [], 'val_auc': []}
best_acc = 0.0; start = time.time()
print(f'\nTraining | AMP={use_amp} | Batch={BATCH_SIZE} | Epochs={NUM_EPOCHS} | Triplets={NUM_TRIPLETS:,}')

for epoch in range(1, NUM_EPOCHS + 1):
    train_ds.trip = train_ds._gen()
    model.train(); ep_loss = ep_valid = 0.0
    pbar = tqdm(train_loader, desc=f'Ep{epoch:02d}/{NUM_EPOCHS}', leave=False)
    for a, p, n in pbar:
        a, p, n = a.to(device, non_blocking=True), p.to(device, non_blocking=True), n.to(device, non_blocking=True)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast(enabled=use_amp):
            loss, valid = criterion(model(a), model(p), model(n))
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        scaler.step(optimizer); scaler.update()
        ep_loss += loss.item(); ep_valid += valid
        pbar.set_postfix(loss=f'{loss.item():.4f}', valid=f'{valid:.1%}')

    al = ep_loss / len(train_loader)
    val_acc, val_auc, val_thr = evaluate(model, val_loader)
    lr = scheduler.get_last_lr()[0]; scheduler.step()
    history['loss'].append(al); history['val_acc'].append(val_acc); history['val_auc'].append(val_auc)
    print(f'Ep{epoch:02d} loss:{al:.4f} ValAcc:{val_acc:.4f} AUC:{val_auc:.4f} thr:{val_thr:.3f} lr:{lr:.2e} {(time.time()-start)/60:.1f}m')

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(), 'best_acc': best_acc},
                   CHECKPOINT / 'facenet_best.pth')
        print(f'   Best saved (acc={best_acc:.4f})')

print(f'\nDone! Best={best_acc:.4f} | {(time.time()-start)/60:.1f}m')

# ── Plot ──────────────────────────────────────────────────────────────────────
ep = range(1, NUM_EPOCHS + 1)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(ep, history['loss'], 'b-o'); axes[0].set_title('Train Loss'); axes[0].grid(0.3)
axes[1].plot(ep, [v*100 for v in history['val_acc']], 'r-o'); axes[1].set_title('Test Accuracy %'); axes[1].grid(0.3)
axes[2].plot(ep, history['val_auc'], 'g-o'); axes[2].set_title('Test AUC'); axes[2].grid(0.3)
plt.suptitle(f'FaceNet CASIA (70/30 split) | Best={best_acc:.4f}', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.savefig(PLOTS_DIR / 'training.png', dpi=150); plt.show()